Extra suppplementary info

In [8]:
# supplementary_information/build_boundary_summary.py

import os
import pandas as pd
from cobra.io import read_sbml_model

# Optional: use your existing merge utility for more robust matching.
# If you have THG/functions/ available, uncomment these two lines:
import sys
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
# from functions.functions_merge_metabolic_networks import network_metabolites_merge_3

# ------------------------
# Paths
# ------------------------
try:
    CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive sessions (e.g., Jupyter)
    CURRENT_DIR = os.getcwd()
    
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
FILES_DIR  = os.path.join(PROJECT_ROOT, "files")

THG_MODEL_PATH       = os.path.join(MODELS_DIR, "THG-beta2_endoA.xml")
IEC_MODEL_PATH       = os.path.join(MODELS_DIR, "EC_model_with_KEGG.xml")        # iEC3006
AFTER_MODEL_PATH     = os.path.join(MODELS_DIR, "THG_endoA_boundary.xml")        # after step 2
COMMON_RS_FILE       = os.path.join(FILES_DIR,  "common_rs.txt")                 # list of new-model IDs

OUTPUT_XLSX = os.path.join(CURRENT_DIR, "Boundary_reaction_summary.xlsx")

# ------------------------
# Helpers
# ------------------------
def core_id_from_model(met_id: str, model) -> str:
    """
    Return a compartment-free 'core' metabolite string for ID formats:
      - met_core_c  (underscore)
      - met_core[c] (brackets)
      - met_corec   (flat suffix; last char(s) is the compartment)
    """
    try:
        comp = model.metabolites.get_by_id(met_id).compartment
    except KeyError:
        return met_id
    if met_id.endswith(f"_{comp}"):        # underscore
        return met_id[:-(len(comp) + 1)]
    if met_id.endswith(f"[{comp}]"):       # brackets
        return met_id[:-(len(comp) + 2)]
    if met_id.endswith(comp):              # flat suffix
        return met_id[:-len(comp)]
    return met_id

def is_single_met_boundary(r):
    return len(r.metabolites) == 1 and r.id in {rx.id for rx in r.model.boundary}

def extract_boundary_df(model, rxn_ids=None):
    if rxn_ids is None:
        rxns = [r for r in model.boundary if is_single_met_boundary(r)]
    else:
        rxns = []
        for rid in rxn_ids:
            if rid in model.reactions and is_single_met_boundary(model.reactions.get_by_id(rid)):
                rxns.append(model.reactions.get_by_id(rid))
    rows = []
    for r in rxns:
        met, coef = next(iter(r.metabolites.items()))
        rows.append({
            "Reaction ID": r.id,
            "Reaction name": r.name,
            "Equation": r.reaction,
            "Lower bound": r.lower_bound,
            "Upper bound": r.upper_bound,
            "Metabolite ID": met.id,
            "Metabolite name": met.name,
            "Metabolite compartment": met.compartment,
            "Stoichiometry": coef
        })
    return pd.DataFrame(rows)

def make_core_comp_index(model):
    """
    Build an index for single-metabolite boundary reactions:
      key = (core_met_id, compartment) → reaction object
    """
    idx = {}
    for r in model.boundary:
        if not is_single_met_boundary(r):
            continue
        met = next(iter(r.metabolites))
        key = (core_id_from_model(met.id, model), met.compartment)
        # if multiple, keep first (rare)
        idx.setdefault(key, r)
    return idx

# ------------------------
# Load models & IDs
# ------------------------
thg_model   = read_sbml_model(THG_MODEL_PATH)     # THG-EndoA (before step 2)
iec_model   = read_sbml_model(IEC_MODEL_PATH)     # iEC3006
after_model = read_sbml_model(AFTER_MODEL_PATH)   # THG-EndoA after refinement

# common_rs.txt is the set of NEW-model boundary rxns deemed “common”
if os.path.exists(COMMON_RS_FILE):
    with open(COMMON_RS_FILE, "r") as f:
        common_rxns_new_ids = [ln.strip() for ln in f if ln.strip()]
else:
    common_rxns_new_ids = []

# ------------------------
# 1) THG-EndoA boundaries (before step 2)
# ------------------------
df_thg_boundary = extract_boundary_df(thg_model)

# ------------------------
# 2) iEC3006 boundaries
# ------------------------
df_iec_boundary = extract_boundary_df(iec_model)

# ------------------------
# 3) Matched boundaries (both models’ bounds & equations)
#    We match by (core metabolite, compartment) to fetch base-model partner.
#    We only report those that are present in the AFTER model & on the common list.
# ------------------------
idx_base = make_core_comp_index(iec_model)
rows_matched = []
for rid in common_rxns_new_ids:
    if rid not in after_model.reactions:
        continue
    r_new = after_model.reactions.get_by_id(rid)
    if not is_single_met_boundary(r_new):
        continue
    met_new = next(iter(r_new.metabolites))
    key = (core_id_from_model(met_new.id, after_model), met_new.compartment)
    r_base = idx_base.get(key, None)
    rows_matched.append({
        # new/after (THG-EndoA after step 2)
        "New Reaction ID": r_new.id,
        "New name": r_new.name,
        "New equation": r_new.reaction,
        "New LB": r_new.lower_bound,
        "New UB": r_new.upper_bound,
        "New metabolite": met_new.id,
        "New compartment": met_new.compartment,
        # base (iEC3006)
        "Base Reaction ID": (r_base.id if r_base else None),
        "Base name": (r_base.name if r_base else None),
        "Base equation": (r_base.reaction if r_base else None),
        "Base LB": (r_base.lower_bound if r_base else None),
        "Base UB": (r_base.upper_bound if r_base else None),
    })
df_matched = pd.DataFrame(rows_matched)

# ------------------------
# 4) THG-EndoA boundaries AFTER the refinement (post step 2)
# ------------------------
df_after_boundary = extract_boundary_df(after_model)

# ------------------------
# 5) Added reactions in step 2 (boundary reactions present after, not before)
# ------------------------
after_ids = {r.id for r in after_model.boundary if is_single_met_boundary(r)}
before_ids = {r.id for r in thg_model.boundary if is_single_met_boundary(r)}
added_ids = sorted(list(after_ids - before_ids))
df_added = extract_boundary_df(after_model, rxn_ids=added_ids)

# ------------------------
# Save Excel
# ------------------------
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_thg_boundary.to_excel(writer, sheet_name="THG-EndoA boundaries", index=False)
    df_iec_boundary.to_excel(writer, sheet_name="iEC3006 boundaries", index=False)
    df_matched.to_excel(writer, sheet_name="Matched boundaries", index=False)
    df_after_boundary.to_excel(writer, sheet_name="After refinement", index=False)
    df_added.to_excel(writer, sheet_name="Added reactions", index=False)

print(f"Excel written to: {OUTPUT_XLSX}")


Excel written to: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\supplementary_material\Boundary_reaction_summary.xlsx


In [3]:
import os
import sys
import pandas as pd
from cobra.io import read_sbml_model


try:
    CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
MODELS_DIR   = os.path.join(PROJECT_ROOT, "models")
FILES_DIR    = os.path.join(PROJECT_ROOT, "files")

THG_MODEL_PATH   = os.path.join(MODELS_DIR, "THG-beta2_endoA.xml")        # before step 2
IEC_MODEL_PATH   = os.path.join(MODELS_DIR, "EC_model_with_KEGG.xml")     # base/iEC3006
AFTER_MODEL_PATH = os.path.join(MODELS_DIR, "THG_endoA_boundary.xml")     # after step 2
COMMON_RS_FILE   = os.path.join(FILES_DIR,  "common_rs.txt")

OUTPUT_XLSX = os.path.join(CURRENT_DIR, "Boundary_reaction_summary.xlsx")


if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from functions.functions_merge_metabolic_networks import network_metabolites_merge_3


def is_single_met_boundary(r):
    """True for single-metabolite boundary reactions in a COBRA model."""
    try:
        return len(r.metabolites) == 1 and r.id in {rx.id for rx in r.model.boundary}
    except Exception:
        return False

def core_id_from_model(met_id: str, model) -> str:
    """
    Compartment-agnostic metabolite 'core' (works for _c, [c], or ...c suffix).
    """
    try:
        comp = model.metabolites.get_by_id(met_id).compartment
    except KeyError:
        return met_id
    if met_id.endswith(f"_{comp}"):       # underscore style
        return met_id[:-(len(comp) + 1)]
    if met_id.endswith(f"[{comp}]"):      # bracket style
        return met_id[:-(len(comp) + 2)]
    if met_id.endswith(comp):             # flat suffix
        return met_id[:-len(comp)]
    return met_id

def extract_boundary_df(model, rxn_ids=None):
    """
    Return a DataFrame of boundary reactions (single-met), including equation & bounds.
    Optionally restrict to a set of reaction IDs.
    """
    if rxn_ids is None:
        rxns = [r for r in model.boundary if is_single_met_boundary(r)]
    else:
        rxns = []
        for rid in rxn_ids:
            if rid in model.reactions and is_single_met_boundary(model.reactions.get_by_id(rid)):
                rxns.append(model.reactions.get_by_id(rid))

    rows = []
    for r in rxns:
        met, coef = next(iter(r.metabolites.items()))
        rows.append({
            "Reaction ID": r.id,
            "Reaction name": r.name,
            "Equation": r.reaction,
            "Lower bound": r.lower_bound,
            "Upper bound": r.upper_bound,
            "Metabolite ID": met.id,
            "Metabolite name": met.name,
            "Metabolite compartment": met.compartment,
            "Stoichiometry": coef
        })
    return pd.DataFrame(rows)

def make_core_comp_index(model):
    """
    Index single-metabolite boundary reactions by (core_met_id, compartment) → reaction.
    If multiple reactions share the same key, the first is kept (we’ll deduplicate output).
    """
    idx = {}
    for r in model.boundary:
        if not is_single_met_boundary(r):
            continue
        met = next(iter(r.metabolites))
        key = (core_id_from_model(met.id, model), met.compartment)
        idx.setdefault(key, r)
    return idx

# Load models & list of common new IDs
thg_model   = read_sbml_model(THG_MODEL_PATH)     # THG-EndoA before Step 2
iec_model   = read_sbml_model(IEC_MODEL_PATH)     # iEC3006 (base)
after_model = read_sbml_model(AFTER_MODEL_PATH)   # THG-EndoA after Step 2

if os.path.exists(COMMON_RS_FILE):
    with open(COMMON_RS_FILE, "r") as f:
        common_rxns_new_ids = [ln.strip() for ln in f if ln.strip()]
else:
    common_rxns_new_ids = []

# Build metabolite equivalence , pass (new, base) in this order 
_, eq_meta = network_metabolites_merge_3(after_model.copy(), iec_model.copy())
# eq_meta: sequence of (new_met_id, base_met_id)

# Filter out pairs where either ID is blank/None
eq_meta = [(new_id, base_id) for new_id, base_id in eq_meta 
           if new_id not in (None, "", "nan", " ") and base_id not in (None, "", "nan", " ")]


# Build mapping by *core* names
base_core_to_new_core = {}
new_core_to_base_core = {}
for new_id, base_id in eq_meta:
    b_core = core_id_from_model(base_id, iec_model)
    n_core = core_id_from_model(new_id, after_model)
    # keep first mapping encountered
    base_core_to_new_core.setdefault(b_core, n_core)
    new_core_to_base_core.setdefault(n_core, b_core)

# Precompute boundary indices
idx_base  = make_core_comp_index(iec_model)
idx_after = make_core_comp_index(after_model)

# ---------------------------------------
# 1) THG-EndoA boundaries (before Step 2)
# ---------------------------------------
df_thg_boundary = extract_boundary_df(thg_model)

# ---------------------------------------
# 2) iEC3006 boundaries
# ---------------------------------------
df_iec_boundary = extract_boundary_df(iec_model)

# ---------------------------------------
# 3) Matched boundaries (both models’ info)
#    Use common_rxns_new_ids if available;
#    otherwise list all keys present in idx_after.
#    Deduplicate by (new_core, compartment) so we show only one row per metabolite/compartment.
# ---------------------------------------
if common_rxns_new_ids:
    after_singlemet = {
        rid for rid in common_rxns_new_ids
        if rid in after_model.reactions and is_single_met_boundary(after_model.reactions.get_by_id(rid))
    }
else:
    after_singlemet = {r.id for r in after_model.boundary if is_single_met_boundary(r)}

rows_matched = []
seen_key = set()  # (new_core, comp) to ensure unique metabolite-compartment entries

for rid in sorted(after_singlemet):
    r_new = after_model.reactions.get_by_id(rid)
    met_new = next(iter(r_new.metabolites))
    new_core = core_id_from_model(met_new.id, after_model)
    comp = met_new.compartment
    dedup_key = (new_core, comp)
    if dedup_key in seen_key:
        continue

    # Find the corresponding base core via the SAME mapping
    base_core = new_core_to_base_core.get(new_core)
    r_base = None
    if base_core is not None:
        # Locate the base boundary rxn on the same (core, compartment)
        r_base = idx_base.get((base_core, comp), None)
    else: continue

    if r_base is None: continue 

    rows_matched.append({
        # After/new model (THG-EndoA after Step 2)
        "THG Reaction ID": r_new.id,
        "THG name": r_new.name,
        "THG LB": r_new.lower_bound,
        "THG UB": r_new.upper_bound,
        "THG metabolite": met_new.id,
        "THG core": new_core,
        "Compartment": comp,
        # Base model (iEC3006)
        "iEC3006 Reaction ID": (r_base.id if r_base else None),
        "iEC3006 name": (r_base.name if r_base else None),
        "iEC3006 equation": (r_base.reaction if r_base else None),
        "iEC3006 LB": (r_base.lower_bound if r_base else None),
        "iEC3006 UB": (r_base.upper_bound if r_base else None),
        "iEC3006 core": (base_core if base_core else None)
    })
    seen_key.add(dedup_key)

df_matched = pd.DataFrame(rows_matched)

# ---------------------------------------
# 4) THG-EndoA boundaries after Step 2
# ---------------------------------------
df_after_boundary = extract_boundary_df(after_model)

# ---------------------------------------
# 5) Added reactions 
# ---------------------------------------
after_ids  = {r.id for r in after_model.boundary if is_single_met_boundary(r)}
before_ids = {r.id for r in thg_model.boundary   if is_single_met_boundary(r)}
added_ids  = sorted(list(after_ids - before_ids))
df_added   = extract_boundary_df(after_model, rxn_ids=added_ids)

# ---------------------------------------
# Write Excel
# ---------------------------------------
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_thg_boundary.to_excel(writer, sheet_name="THG-EndoA boundaries", index=False)
    df_iec_boundary.to_excel(writer, sheet_name="iEC3006 boundaries", index=False)
    df_matched.to_excel(writer,      sheet_name="Matched boundaries", index=False)
    df_after_boundary.to_excel(writer, sheet_name="After refinement", index=False)
    df_added.to_excel(writer,        sheet_name="Added reactions", index=False)

print(f"Excel written to: {OUTPUT_XLSX}")


Read LP format model from file C:\Users\MFRA0106\AppData\Local\Temp\tmpu7a4l8jg.lp
Reading time = 0.08 seconds
: 17163 rows, 46334 columns, 211096 nonzeros
Read LP format model from file C:\Users\MFRA0106\AppData\Local\Temp\tmpgw275anh.lp
Reading time = 0.01 seconds
: 2114 rows, 6012 columns, 25478 nonzeros
equivalent metabolites between n1 and n2:  638
Excel written to: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\supplementary_material\Boundary_reaction_summary.xlsx


Step 3 (putative transport reaction output)

In [6]:
# supplementary_information/make_deadends_and_TR_summary.py

import os
import sys
import pandas as pd
from cobra.io import read_sbml_model

# -----------------------
# Paths
# -----------------------
try:
    CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
MODELS_DIR   = os.path.join(PROJECT_ROOT, "models")
FILES_DIR    = os.path.join(PROJECT_ROOT, "files")

# Final model after round-2 TRs (adjust if needed)
MODEL_FINAL_PATH = os.path.join(MODELS_DIR, "model_full_THG_round2.xml")

# Round-2 pickle artifacts you saved during TR inference
PKL_INIT_S = os.path.join(FILES_DIR, "DeadEnd_S_THG_round2.pkl")            # initial dead-end substrates (before round-2 TRs)
PKL_INIT_P = os.path.join(FILES_DIR, "DeadEnd_P_THG_round2.pkl")            # initial dead-end products (before round-2 TRs)
PKL_FINAL  = os.path.join(FILES_DIR, "Total_DeadEnd_M_left_round2.pkl")     # final dead-ends (after round-2 TRs)
PKL_TR_ALL = os.path.join(FILES_DIR, "final_reactions_added_THG_round2.pkl")# all TRs added up to round 2

OUTPUT_XLSX = os.path.join(CURRENT_DIR, "Supplementary_deadends_and_TRs.xlsx")

# -----------------------
# Utilities
# -----------------------
def core_id(met_id: str, model) -> str:
    """Return compartment-free metabolite core ID for styles *_c*, *[c]*, or flat suffix *...c*."""
    try:
        comp = model.metabolites.get_by_id(met_id).compartment
    except KeyError:
        return met_id
    if met_id.endswith(f"_{comp}"):
        return met_id[:-(len(comp) + 1)]
    if met_id.endswith(f"[{comp}]"):
        return met_id[:-(len(comp) + 2)]
    if met_id.endswith(comp):
        return met_id[:-len(comp)]
    return met_id

def _normalize_ann(value):
    """Return annotation value as string; join lists/tuples/sets with ';'."""
    if value is None:
        return None
    if isinstance(value, (list, tuple, set)):
        return ";".join(str(v) for v in value)
    return str(value)

def get_first_ann(annotation: dict, *keys):
    """
    Return first non-empty annotation among the provided keys.
    Example: get_first_ann(r.annotation, 'vmhreaction', 'vmh.reaction')
    """
    for k in keys:
        if k in annotation and annotation[k]:
            return _normalize_ann(annotation[k])
    return None

def met_row(model, met_id):
    """Build a dict with metabolite fields + external IDs, skipping if met not in model."""
    if met_id not in model.metabolites:
        return None
    m = model.metabolites.get_by_id(met_id)
    ann = m.annotation or {}

    return {
        "Metabolite ID": m.id,
        "Metabolite name": m.name,
        "Formula": getattr(m, "formula", None),
        "Compartment": m.compartment,
        "Core": core_id(m.id, model),

        # Metabolite external IDs
        "BiGG metabolite":   get_first_ann(ann, "bigg.metabolite"),
        "KEGG compound":     get_first_ann(ann, "kegg.compound"),
        "ChEBI":             get_first_ann(ann, "chebi"),
        "VMH metabolite":    get_first_ann(ann, "vmhmetabolite", "vmh.metabolite"),
        "MetaNetX chemical": get_first_ann(ann, "metanetx.chemical"),
        "LIPID MAPS":        get_first_ann(ann, "lipidmaps"),
        "PubChem CID":       get_first_ann(ann, "pubchem.compound"),
        "InChIKey":          get_first_ann(ann, "inchikey"),
        "InChI":             get_first_ann(ann, "inchi"),
    }

def classify_deadend_in_final_model(model, met_ids):
    """
    For each metabolite, check all reactions that include it.
    If stoich sign is always >0 -> final product; always <0 -> final substrate.
    If both signs appear (shouldn't happen for a true dead-end), mark 'Both'.
    """
    finals_S, finals_P, finals_Both, not_found = [], [], [], []
    for mid in set(met_ids):
        if mid not in model.metabolites:
            not_found.append(mid)
            continue
        m = model.metabolites.get_by_id(mid)
        neg = pos = False
        for rxn in m.reactions:
            coef = rxn.metabolites[m]
            if coef < 0: neg = True
            elif coef > 0: pos = True
        if neg and not pos:
            finals_S.append(mid)
        elif pos and not neg:
            finals_P.append(mid)
        elif pos and neg:
            finals_Both.append(mid)
        else:
            finals_Both.append(mid)
    return finals_S, finals_P, finals_Both, not_found

def reaction_ann_cols(r):
    """Return reaction external IDs as dict columns."""
    ann = r.annotation or {}
    return {
        "KEGG reaction":      get_first_ann(ann, "kegg.reaction"),
        "BiGG reaction":      get_first_ann(ann, "bigg.reaction"),
        "MetaNetX reaction":  get_first_ann(ann, "metanetx.reaction"),
        "VMH reaction":       get_first_ann(ann, "vmhreaction", "vmh.reaction"),
    }

def met_ann_cols(m):
    """Return metabolite external IDs as dict columns (for TR sheet)."""
    ann = m.annotation or {}
    return {
        "BiGG metabolite":   get_first_ann(ann, "bigg.metabolite"),
        "KEGG compound":     get_first_ann(ann, "kegg.compound"),
        "ChEBI":             get_first_ann(ann, "chebi"),
        "VMH metabolite":    get_first_ann(ann, "vmhmetabolite", "vmh.metabolite"),
        "MetaNetX chemical": get_first_ann(ann, "metanetx.chemical"),
        "LIPID MAPS":        get_first_ann(ann, "lipidmaps"),
        "PubChem CID":       get_first_ann(ann, "pubchem.compound"),
        "InChIKey":          get_first_ann(ann, "inchikey"),
        "InChI":             get_first_ann(ann, "inchi"),
    }

def tr_row(model, rxn_id):
    """Build a row for a transport reaction: reaction info and the two metabolites it connects."""
    if rxn_id not in model.reactions:
        return None
    r = model.reactions.get_by_id(rxn_id)
    mets = list(r.metabolites.items())  # [(met, coef), ...]

    # Grab up to two metabolites (transport should be 2, but we handle odd cases)
    metA, coefA = (mets[0][0], float(mets[0][1])) if len(mets) >= 1 else (None, None)
    metB, coefB = (mets[1][0], float(mets[1][1])) if len(mets) >= 2 else (None, None)

    # Direction hint from stoichiometry if exactly two mets and opposite signs.
    direction = None
    if coefA is not None and coefB is not None:
        if coefA < 0 and coefB > 0:
            direction = f"{metA.compartment} → {metB.compartment}"
        elif coefA > 0 and coefB < 0:
            direction = f"{metB.compartment} → {metA.compartment}"

    # Safely collect annotation dicts
    rxn_ann = reaction_ann_cols(r)
    metA_ann = met_ann_cols(metA) if metA else {}
    metB_ann = met_ann_cols(metB) if metB else {}

    row = {
        # Reaction info
        "Reaction ID": r.id,
        "Reaction name": r.name,
        "Lower bound": r.lower_bound,
        "Upper bound": r.upper_bound,
        "Equation": r.reaction,
        **rxn_ann,

        # MetA info
        "MetA ID": (metA.id if metA else None),
        "MetA name": (metA.name if metA else None),
        "MetA formula": (getattr(metA, "formula", None) if metA else None),
        "MetA compartment": (metA.compartment if metA else None),
        "MetA coeff": coefA,
        **{f"MetA {k}": v for k, v in metA_ann.items()},

        # MetB info
        "MetB ID": (metB.id if metB else None),
        "MetB name": (metB.name if metB else None),
        "MetB formula": (getattr(metB, "formula", None) if metB else None),
        "MetB compartment": (metB.compartment if metB else None),
        "MetB coeff": coefB,
        **{f"MetB {k}": v for k, v in metB_ann.items()},

        "Connects (direction)": direction,
    }
    return row


def df_from_met_list(model, met_ids, title):
    rows, skips = [], 0
    for mid in met_ids:
        row = met_row(model, mid)
        if row is None:
            skips += 1
            continue
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"{title}: {len(df)} rows (skipped {skips} not-found IDs)")
    # Order columns if present
    cols = [
        "Metabolite ID", "Metabolite name", "Formula", "Compartment", "Core",
        "BiGG metabolite", "KEGG compound", "ChEBI", "VMH metabolite",
        "MetaNetX chemical", "LIPID MAPS", "PubChem CID", "InChIKey", "InChI"
    ]
    return df[cols] if not df.empty else df

# -----------------------
# Load model & pickles
# -----------------------
if not os.path.exists(MODEL_FINAL_PATH):
    raise FileNotFoundError(f"Final model not found at: {MODEL_FINAL_PATH}")

model = read_sbml_model(MODEL_FINAL_PATH)
print(f"Loaded final model: {MODEL_FINAL_PATH}")
print(f"Model stats: {len(model.reactions)} reactions, {len(model.metabolites)} metabolites")

def load_list(p):
    if not os.path.exists(p):
        print(f"[WARN] Missing file: {p}")
        return []
    try:
        import pickle
        with open(p, "rb") as f:
            data = pickle.load(f)
        if isinstance(data, (list, tuple, set)):
            return list(data)
        print(f"[WARN] Unexpected pickle content type at {p}: {type(data)}")
        return []
    except Exception as e:
        print(f"[WARN] Failed to load {p}: {e}")
        return []

initial_S_ids = load_list(PKL_INIT_S)
initial_P_ids = load_list(PKL_INIT_P)
final_dead_ids = load_list(PKL_FINAL)
tr_ids_all     = load_list(PKL_TR_ALL)

print(f"Initial dead-end substrates (pickle): {len(initial_S_ids)}")
print(f"Initial dead-end products   (pickle): {len(initial_P_ids)}")
print(f"Final dead-ends after TRs   (pickle): {len(final_dead_ids)}")
print(f"Transport reactions added   (pickle): {len(tr_ids_all)}")

# -----------------------
# Build DataFrames
# -----------------------

# Initial lists (from pickles)
df_init_S = df_from_met_list(model, initial_S_ids, "Initial dead-end substrates")
df_init_P = df_from_met_list(model, initial_P_ids, "Initial dead-end products")

# Final: split the combined final list by inspecting stoichiometry in the FINAL model
final_S_ids, final_P_ids, final_both, final_not_found = classify_deadend_in_final_model(model, final_dead_ids)
if final_both:
    print(f"[NOTE] {len(final_both)} final dead-ends appear as both substrate and product (unexpected for true dead-ends).")
if final_not_found:
    print(f"[NOTE] {len(final_not_found)} final dead-end IDs not found in the final model.")

df_final_S = df_from_met_list(model, final_S_ids, "Final dead-end substrates")
df_final_P = df_from_met_list(model, final_P_ids, "Final dead-end products")

# Transport reactions added (+ reaction & metabolite annotations)
tr_rows, tr_skips = [], 0
for rid in tr_ids_all:
    row = tr_row(model, rid)
    if row is None:
        tr_skips += 1
        continue
    tr_rows.append(row)
df_tr = pd.DataFrame(tr_rows)
print(f"Transport reactions added: {len(df_tr)} rows (skipped {tr_skips} not-found IDs)")

# Order columns for TR sheet if present
if not df_tr.empty:
    tr_cols = [
        "Reaction ID","Reaction name","Lower bound","Upper bound","Equation",
        "KEGG reaction","BiGG reaction","MetaNetX reaction","VMH reaction",
        "MetA ID","MetA name","MetA formula","MetA compartment","MetA coeff",
        "MetA BiGG metabolite","MetA KEGG compound","MetA ChEBI","MetA VMH metabolite",
        "MetA MetaNetX chemical","MetA LIPID MAPS","MetA PubChem CID","MetA InChIKey","MetA InChI",
        "MetB ID","MetB name","MetB formula","MetB compartment","MetB coeff",
        "MetB BiGG metabolite","MetB KEGG compound","MetB ChEBI","MetB VMH metabolite",
        "MetB MetaNetX chemical","MetB LIPID MAPS","MetB PubChem CID","MetB InChIKey","MetB InChI",
        "Connects (direction)"
    ]
    # Keep only columns that exist (some IDs may be missing)
    tr_cols = [c for c in tr_cols if c in df_tr.columns]
    df_tr = df_tr[tr_cols]

# -----------------------
# Save Excel
# -----------------------
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_init_S.to_excel(writer, sheet_name="Initial dead-end substrates", index=False)
    df_init_P.to_excel(writer, sheet_name="Initial dead-end products",   index=False)
    df_final_S.to_excel(writer, sheet_name="Final dead-end substrates",  index=False)
    df_final_P.to_excel(writer, sheet_name="Final dead-end products",    index=False)
    df_tr.to_excel(writer,     sheet_name="Transport reactions added",   index=False)

print(f"\nExcel written to: {OUTPUT_XLSX}")


Loaded final model: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\models\model_full_THG_round2.xml
Model stats: 24388 reactions, 17163 metabolites
Initial dead-end substrates (pickle): 1811
Initial dead-end products   (pickle): 2826
Final dead-ends after TRs   (pickle): 4454
Transport reactions added   (pickle): 1221
Initial dead-end substrates: 1811 rows (skipped 0 not-found IDs)
Initial dead-end products: 2826 rows (skipped 0 not-found IDs)
Final dead-end substrates: 1722 rows (skipped 0 not-found IDs)
Final dead-end products: 2732 rows (skipped 0 not-found IDs)
Transport reactions added: 1221 rows (skipped 0 not-found IDs)

Excel written to: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\supplementary_material\Supplementary_deadends_and_TRs.xlsx


Information of extra boundary, sink and demand rs added in the Refinement of compartment connectivity and protected fluxes for transcriptomics integration step

In [7]:
# supplementary_information/export_protected_reactions.py

import os
import sys
import pandas as pd
from cobra.io import read_sbml_model

# ------------------------
# Paths
# ------------------------
try:
    CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
MODELS_DIR   = os.path.join(PROJECT_ROOT, "models")
FILES_DIR    = os.path.join(PROJECT_ROOT, "files")

MODEL_PATH   = os.path.join(MODELS_DIR, "model_full_THG_optimized_sinks_demands.xml")
PROTECTED_TXT= os.path.join(FILES_DIR,  "unique_ids_protected_rs_endoA.txt")
OUTPUT_XLSX  = os.path.join(CURRENT_DIR, "Protected_reactions_summary.xlsx")

# Make project code importable if needed
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# ------------------------
# Helpers
# ------------------------
def _norm_anno(val):
    """Normalize annotation field: join lists/tuples, pass strings, else ''."""
    if val is None:
        return ""
    if isinstance(val, (list, tuple, set)):
        try:
            return ";".join(str(x) for x in val if x is not None and str(x) != "")
        except Exception:
            return str(list(val))
    return str(val)

def load_protected_ids(path):
    ids = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            # allow comma-separated or whitespace-separated IDs on a line
            parts = [p.strip() for chunk in s.split(",") for p in chunk.split()]
            ids.extend([p for p in parts if p])
    # de-duplicate, preserve order
    seen = set()
    uniq = []
    for rid in ids:
        if rid not in seen:
            seen.add(rid)
            uniq.append(rid)
    return uniq

# ------------------------
# Load model & IDs
# ------------------------
print(f"Loading model: {MODEL_PATH}")
model = read_sbml_model(MODEL_PATH)

print(f"Loading protected IDs: {PROTECTED_TXT}")
protected_ids = load_protected_ids(PROTECTED_TXT)
print(f"Protected IDs loaded: {len(protected_ids)}")

# ------------------------
# Collect rows
# ------------------------
rows = []
missing = []

for rid in protected_ids:
    if rid not in model.reactions:
        missing.append(rid)
        continue

    rxn = model.reactions.get_by_id(rid)
    ann = rxn.annotation or {}

    rows.append({
        "Reaction ID": rxn.id,
        "Reaction": rxn.reaction,          # human-readable equation
        "Name": rxn.name,
        "kegg.reaction": _norm_anno(ann.get("kegg.reaction")),
        "bigg.reaction": _norm_anno(ann.get("bigg.reaction")),
        "metanetx.reaction": _norm_anno(ann.get("metanetx.reaction")),
        "vmh.reaction": _norm_anno(ann.get("vmhreaction")),
    })

df = pd.DataFrame(rows)

# ------------------------
# Save Excel (with a missing-IDs sheet)
# ------------------------
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Protected reactions", index=False)
    if missing:
        pd.DataFrame({"Missing reaction IDs": missing}).to_excel(
            writer, sheet_name="IDs not in model", index=False
        )

print(f"Excel written to: {OUTPUT_XLSX}")
print(f"Found {len(df)} protected reactions in the model; {len(missing)} IDs were not found.")


Loading model: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\models\model_full_THG_optimized_sinks_demands.xml
Loading protected IDs: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\files\unique_ids_protected_rs_endoA.txt
Protected IDs loaded: 370
Excel written to: c:\Users\MFRA0106\Documents\Putative_TR\03_Add_Transport_reactions\03_Add_Transport_reactions\reorganization_2025\THG\supplementary_material\Protected_reactions_summary.xlsx
Found 370 protected reactions in the model; 0 IDs were not found.
